# 01 · Inspect the raw PhysioNet training data

This notebook inspects the **raw** patient PSV files for the fresh BedAllocation rebuild. It does not reuse the old project's processed features or trained models.

The official PhysioNet `SepsisLabel` is 1 from six hours before annotated sepsis onset onward. The initial target that looked for any positive source label in the next six rows also made post-onset rows positive, so it was unsuitable for early-onset evaluation. The corrected target infers onset as the first source-positive row plus six hours, labels whether onset is in the next six hourly positions, and excludes post-onset rows. See `docs/phase-6-corrected-target.md`.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

CWD = Path.cwd().resolve()
candidates = [CWD, CWD / 'ml-service', CWD.parent / 'ml-service', CWD.parent.parent / 'ml-service']
ML_ROOT = next((candidate for candidate in candidates if (candidate / 'pyproject.toml').exists()), None)
if ML_ROOT is None:
    raise FileNotFoundError('Open this notebook from the repository root or notebooks/ml; cannot locate ml-service/.')
sys.path.insert(0, str(ML_ROOT / 'src'))
from bedallocation_ml.data import list_patient_files, sepsis_onset_target
RAW_DIR = ML_ROOT / 'data' / 'raw'
files = list_patient_files(RAW_DIR)
print(f'Raw patient files found: {len(files):,}')
print(f'Example file: {files[0].relative_to(RAW_DIR)}')

In [ ]:
sample = pd.read_csv(files[0], sep='|')
print('Shape:', sample.shape)
print('Columns:', sample.columns.tolist())
display(sample.head(10))

In [ ]:
label_counts = sample['SepsisLabel'].value_counts(dropna=False).sort_index()
print('SepsisLabel counts in this one patient file:')
display(label_counts.to_frame('rows'))
print('Missingness fraction for the first file:')
display(sample.isna().mean().sort_values(ascending=False).head(15).to_frame('missing_fraction'))

In [ ]:
HORIZON = 6
target, retained_rows = sepsis_onset_target(sample['SepsisLabel'].to_numpy(), horizon=HORIZON)
print(f'Rows in source file: {len(sample)}')
print(f'Retained pre-onset/fully-observed rows: {retained_rows}')
if len(target):
    print('Positive onset-within-horizon rows:', int(target.sum()))
    print('Positive fraction among retained rows:', float(target.mean()))
else:
    print('This patient file is too short to provide retained rows for the selected horizon.')

## Before moving on

- Confirm the file headers and source label values are consistent.
- Review missingness and class imbalance; missing measurements are not normal readings.
- The preprocessing script splits at the **patient level**, so one patient cannot occur in multiple splits.
- Read `docs/phase-6-corrected-target.md`. The target is a retrospective research target, not ICU admission probability or clinical authorization.